# 🌙 SleepWise - 01. Exploratory Data Analysis (EDA) & Preprocessing
**Fitur:** Dataset & Preprocessing  
**Branch:** `feature/01-data-preprocessing`  
Notebook ini mencakup pemuatan dataset, pembersihan missing values & duplikat, feature engineering (split Blood Pressure), EDA visualisasi, pemetaan 3 kategori (Good/Average/Poor), dan pipeline encoding.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

sns.set_theme(style="whitegrid", palette="muted")
df = pd.read_csv("../data/raw/Sleep_health_and_lifestyle_dataset.csv")
df.head()

## 1. Pembersihan Data (Missing Values & Standarisasi)

In [ ]:
# 1. Hapus Person ID
if "Person ID" in df.columns:
    df.drop(columns=["Person ID"], inplace=True)

# 2. Imputasi Sleep Disorder
df["Sleep Disorder"] = df["Sleep Disorder"].fillna("None")

# 3. Standarisasi BMI Category
df["BMI Category"] = df["BMI Category"].replace({"Normal Weight": "Normal"})

# 4. Split Blood Pressure menjadi Sistolik dan Diastolik
bp_split = df["Blood Pressure"].astype(str).str.split("/", expand=True)
df["Systolic_BP"] = bp_split[0].astype(int)
df["Diastolic_BP"] = bp_split[1].astype(int)
df.drop(columns=["Blood Pressure"], inplace=True)

# 5. Label Target 3 Kategori (Poor, Average, Good)
def map_sleep(score):
    if score <= 5: return "Poor"
    elif score == 6: return "Average"
    else: return "Good"

df["Sleep_Quality_Class"] = df["Quality of Sleep"].apply(map_sleep)
print("Distribusi Target:")
print(df["Sleep_Quality_Class"].value_counts())
df.info()

## 2. Visualisasi EDA (Distribusi & Korelasi)

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(x="Sleep_Quality_Class", data=df, palette=["#fbbf24", "#4ade80", "#f87171"], order=["Average", "Good", "Poor"])
plt.title("Distribusi Kategori Kualitas Tidur")
plt.show()

plt.figure(figsize=(10, 6))
numeric_cols = df.select_dtypes(include=[np.number]).columns
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap="Blues", fmt=".2f")
plt.title("Korelasi Fitur Numerik")
plt.show()

## 3. Ekspor Data Bersih & ColumnTransformer

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

cat_cols = ["Gender", "Occupation", "BMI Category", "Sleep Disorder"]
num_cols = ["Age", "Sleep Duration", "Physical Activity Level", "Stress Level", "Heart Rate", "Daily Steps", "Systolic_BP", "Diastolic_BP"]

X = df[cat_cols + num_cols]
y = df["Sleep_Quality_Class"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
    ("num", StandardScaler(), num_cols)
])
preprocessor.fit(X_train)
print("Preprocessing pipeline berhasil dilatih!")